# IVtrace v1.0a-jupyter #

Программное обеспечение IVtrace предназначено для автоматизированного снятия амплитудной характеристики датчиков тока. Реализовано на языке Python в среде Jupyter Notebook. Использует библиотеки pyvisa для взаимодействия с измерительными приборами по стандарту SCPI через интерфейс NI‑VISA, pandas для накопления и экспорта данных, matplotlib для визуализации. ПО поддерживает:

- автоматическое обнаружение вольтметра (АКИП‑2101 / Siglent) и источника тока (ITECH IT‑M3910D);

- кэширование параметров измерения в JSON‑конфигурации;

- импульсный режим работы источника (включение/выключение выхода на каждой токовой точке);

- ручное переключение полярности с логическим учётом знака в результатах;

- сохранение метаданных и измеренных значений в формат CSV.

Для работы предварительно требуется установить NI-VISA (http://www.ni.com/download/ni-visa-17.5/7220/en/), а также, очевидно, python и pip

### Установка библиотек ###

In [ ]:
%pip install pyvisa pandas matplotlib

### Импорт и проверка библиотек ###

In [8]:
import os
import sys
import time
import json
from datetime import datetime
from pathlib import Path
import pyvisa
import pandas as pd
import matplotlib.pyplot as plt

print("PyVISA:", pyvisa.__version__)
print("Pandas:", pd.__version__)
print("Все библиотеки готовы.")

PyVISA: 1.16.2
Pandas: 2.3.3
Все библиотеки готовы.


### Автоматическое определение устройств ###

1. Создаётся менеджер ресурсов pyvisa.ResourceManager() (без аргумента '@py' — используется системная NI‑VISA).
2. Получается список всех доступных VISA‑ресурсов. Если список пуст — работа скрипта аварийно завершается.
3. Для каждого ресурса:
- открывается сессия, задаётся кодировка utf-8 и таймаут 3 с;
- отправляется команда *IDN?, ответ анализируется на наличие ключевых слов (AKIP-2101 / SIGLENT → вольтметр; ITECH / IT-M → источник тока);
- сессия закрывается.
4. Если оба адреса не найдены — аварийное завершение.
5. Найденные адреса dmm_addr и curr_src_addr сохраняются и выводятся пользователю.

In [9]:
rm = pyvisa.ResourceManager()
resources = rm.list_resources()

if len(resources) == 0:
    raise SystemExit("Не найдено ни одного VISA-ресурса. Проверьте подключение и драйверы.")

dmm_addr = None
curr_src_addr = None

for res in resources:
    try:
        instr = rm.open_resource(res)
        instr.encoding = 'utf-8'
        instr.timeout = 3000
        idn = instr.query('*IDN?').strip()
        print(f'{res}  ->  {idn}')
        if 'AKIP-2101' in idn or 'SIGLENT' in idn.upper():
            dmm_addr = res
        elif 'ITECH' in idn.upper() or 'IT-M' in idn.upper():
            curr_src_addr = res
        instr.close()
    except Exception as e:
        print(f'{res}  ->  Ошибка при опросе: {e}')

if not dmm_addr or not curr_src_addr:
    raise SystemExit("Не удалось обнаружить оба прибора. Проверьте список ресурсов выше.")

print(f"\nВольтметр: {dmm_addr}")
print(f"Источник тока: {curr_src_addr}")

USB0::0xF4EC::0x1201::SDM35HBQ7R1075::INSTR  ->  AKIP,AKIP-2101,SDM35HBQ7R1075,1.02.01.27R2
USB0::0x2EC7::0x3900::805935011797840004::INSTR  ->  ITECH,IT-M3910D-10-1020,805935011797840004,00.01.1701,411.R,170.R

Вольтметр: USB0::0xF4EC::0x1201::SDM35HBQ7R1075::INSTR
Источник тока: USB0::0x2EC7::0x3900::805935011797840004::INSTR


### Инициализация ###

1. Определяется корневая директория сохранения (C:/IVTraceData), при необходимости создаётся. В ней же хранится файл ivtrace_config.json для кэширования настроек.
2. Реализованы функции load_config() и save_config() для чтения/записи параметров в JSON.
При запуске:
- загружается предыдущий конфиг (если есть);
- пользователю отображаются сохранённые параметры и предлагается их использовать (ввод y/n);
- при отказе или отсутствии конфига запрашиваются: начальный ток, конечный ток, шаг, ограничение напряжения, задержка между установкой и измерением;
- ввод значения осуществляется в цикле с обработкой ошибок преобразования типов.
3. Выбор ветви (positive/negative) реализован отдельным диалогом с поддержкой коротких синонимов (p, n, +, -) и подсказкой последнего использованного значения.
4. Полученные параметры немедленно сохраняются в конфигурационный файл.
5. Формируется имя CSV‑файла, включающее временную метку и указанную ветвь, например IVtrace_positive_20250525_143025.csv.

In [15]:
# ============================
# Константа: папка для сохранения (измените при необходимости)
# ============================
SAVE_DIR = Path("C:/IVTraceData")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = SAVE_DIR / "ivtrace_config.json"

# ============================
# Функции загрузки/сохранения конфига
# ============================
def load_config():
    if CONFIG_FILE.exists():
        try:
            with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"Ошибка чтения конфига: {e}")
    return None

def save_config(config):
    try:
        with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=4)
    except Exception as e:
        print(f"Ошибка сохранения конфига: {e}")

# ============================
# Ввод параметров измерения (с кэшированием)
# ============================

print("\n=== Настройка измерения ===")

# Загружаем сохранённые параметры, если есть
saved_config = load_config()
if saved_config:
    print("\nНайдены сохранённые параметры:")
    print(f"  Ток: {saved_config['I_start']} → {saved_config['I_stop']} А, шаг {saved_config['I_step']} А")
    print(f"  Ограничение напряжения: {saved_config['V_limit']} В")
    print(f"  Задержка: {saved_config['delay']} с")
    print(f"  Последняя ветвь: {saved_config.get('direction', '?')}")
    use_prev = input("\nИспользовать эти параметры? (y/n, по умолчанию y): ").strip().lower()
    if use_prev != 'n':
        # берём значения из конфига
        I_start = saved_config['I_start']
        I_stop  = saved_config['I_stop']
        I_step  = saved_config['I_step']
        V_limit = saved_config['V_limit']
        delay   = saved_config['delay']
        print("\nПараметры загружены.")
    else:
        # будем вводить новые
        I_start = I_stop = I_step = V_limit = delay = None
else:
    I_start = I_stop = I_step = V_limit = delay = None

# Если параметры не загружены или пользователь отказался, запрашиваем
if I_start is None:
    while True:
        try:
            I_start = float(input("Начальный ток (А): "))
            I_stop  = float(input("Конечный ток (А): "))
            I_step  = float(input("Шаг по току (А): "))
            V_limit = float(input("Ограничение напряжения на источнике (В): "))
            delay   = float(input("Задержка между установкой и измерением (с): "))
            break
        except ValueError as e:
            print(f"Ошибка ввода: {e}. Попробуйте снова.\n")
    # Сохраняем новые параметры (кроме direction, его обновим позже)
    save_config({
        'I_start': I_start,
        'I_stop': I_stop,
        'I_step': I_step,
        'V_limit': V_limit,
        'delay': delay,
        'direction': ''  # временно, заполним после ввода ветви
    })
else:
    # Если загрузили, всё равно уточним, что direction будем запрашивать отдельно
    pass

# Ввод ветви (с подсказкой последней, если есть)
last_dir = saved_config.get('direction', '') if saved_config else ''
if last_dir:
    hint = f" (Enter для {last_dir}, или введите p/n/+/-)"
else:
    hint = ""

while True:
    dir_input = input(f"Ветвь (positive/p/+ или negative/n/-){hint}: ").strip().lower()
    if dir_input == '' and last_dir:
        dir_input = last_dir
    if dir_input in ('positive', 'p', '+'):
        direction = 'positive'
        break
    elif dir_input in ('negative', 'n', '-'):
        direction = 'negative'
        break
    else:
        print("Некорректная ветвь. Используйте positive/p/+ или negative/n/-")

# Обновляем конфиг с новым направлением (и другими параметрами, если они изменились)
save_config({
    'I_start': I_start,
    'I_stop': I_stop,
    'I_step': I_step,
    'V_limit': V_limit,
    'delay': delay,
    'direction': direction
})

# Формируем имя файла с временной меткой
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = SAVE_DIR / f"IVtrace_{direction}_{timestamp_str}.csv"

print(f"\nФайл результатов: {csv_filename}")
print(f"Диапазон: {I_start}..{I_stop} А, шаг {I_step} А, ограничение V={V_limit} В, ветвь: {direction}")

# ============================
# Подключение и инициализация приборов (без изменений)
# ============================

dmm = rm.open_resource(dmm_addr)
dmm.encoding = 'utf-8'
dmm.timeout = 5000

curr = rm.open_resource(curr_src_addr)
curr.encoding = 'utf-8'
curr.timeout = 5000


=== Настройка измерения ===

Найдены сохранённые параметры:
  Ток: 0.0 → 10.0 А, шаг 1.0 А
  Ограничение напряжения: 3.0 В
  Задержка: 1.0 с
  Последняя ветвь: positive



Использовать эти параметры? (y/n, по умолчанию y):  y



Параметры загружены.


Ветвь (positive/p/+ или negative/n/-) (Enter для positive, или введите p/n/+/-):  



Файл результатов: C:\IVTraceData\IVtrace_positive_20260525_101614.csv
Диапазон: 0.0..10.0 А, шаг 1.0 А, ограничение V=3.0 В, ветвь: positive


### Измерения ###

1. Выполняется сброс источника (`*RST`), пауза 1 с.
2. Устанавливается ограничение напряжения (`SOUR:VOLT:LIM`) и нулевой ток. Выход остаётся выключенным.
3. Определяются знак `sign` (+1 для positive, -1 для negative) и количество шагов  
   `num_steps = int((I_stop - I_start)/I_step) + 1`.
4. Для каждого шага `step` от 0 до `num_steps-1`:
   - вычисляется абсолютный ток `abs_current = I_start + step * I_step` (всегда ≥0);
   - знаковый ток `signed_current = abs_current * sign` (только для записи);
   - на источник подаётся команда `SOUR:CURR {abs_current}`;
   - включается выход (`OUTP ON`);
   - выдерживается задержка `delay` с;
   - производится трёхкратное измерение напряжения вольтметром (`MEAS:VOLT:DC?`), результат усредняется (`v_avg`);
   - выход выключается (`OUTP OFF`);
   - выдерживается задержка `delay` с;
   - в список `results` добавляется запись: время, `I_set_A = signed_current`, `V_meas_V = v_avg`.
5. После цикла источник устанавливает ток 0 и выключает выход (однократно).
6. Из `results` формируется `pandas.DataFrame`.
7. Файл CSV открывается с кодировкой UTF-8. В начало построчно записываются метаданные (каждая строка начинается с `#`): диапазон, шаг, ограничение напряжения, ветвь, задержка, время измерения, количество точек. Затем пустая строка-разделитель `#`.
8. В CSV дописываются данные из `DataFrame` (заголовки столбцов `Timestamp`, `I_set_A`, `V_meas_V`, сами строки).
9. Пользователю выводятся имя файла и первые 10 строк данных.

In [ ]:
# ============================
# Измерительный цикл (импульсный режим)
# ============================

print("Подключаюсь к приборам...\n")

curr.write('*RST')
time.sleep(1)
curr.write(f'SOUR:VOLT:LIM {V_limit}')
curr.write('SOUR:CURR 0')
print("Источник тока готов. Начинаю измерения...\n")

results = []
sign = -1 if direction == 'negative' else 1
num_steps = int((I_stop - I_start) / I_step) + 1

for step in range(num_steps):
    # Вычисляем абсолютный ток (всегда положительный для прибора)
    abs_current = I_start + step * I_step
    # Ток со знаком – только для записи в результаты
    signed_current = abs_current * sign
    
    # Устанавливаем положительный ток на источнике
    curr.write(f'SOUR:CURR {abs_current}')
    # Включаем выход
    curr.write('OUTP ON')
    time.sleep(delay)
    
    # Измеряем напряжение (усреднение 3 раза)
    voltages = []
    for _ in range(3):
        try:
            v = float(dmm.query('MEAS:VOLT:DC?'))
            voltages.append(v)
        except Exception as e:
            print(f"Ошибка измерения: {e}")
    v_avg = sum(voltages) / len(voltages) if voltages else 0.0
    
    # Выключаем выход
    curr.write('OUTP OFF')
    time.sleep(delay)
    
    # Сохраняем результат с учётом знака
    results.append({
        'Timestamp': datetime.now().isoformat(),
        'I_set_A': signed_current,   # уже со знаком
        'V_meas_V': v_avg
    })
    
    print(f"  I = {signed_current:+.4f} А  ->  V = {v_avg:.6f} В")

# После цикла – финальное выключение (один раз)
curr.write('SOUR:CURR 0')
curr.write('OUTP OFF')
print("\nИзмерения завершены, источник выключен.")

# ============================
# Сохранение в CSV (метаданные + данные)
# ============================

df = pd.DataFrame(results)

with open(csv_filename, 'w', encoding='utf-8') as f:
    f.write(f"# Диапазон: {I_start}..{I_stop} А, шаг {I_step} А, ограничение V={V_limit} В, ветвь: {direction}\n")
    f.write(f"# Задержка между установкой и измерением: {delay} с\n")
    f.write(f"# Время измерения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Всего точек: {len(df)}\n")
    f.write("#\n")
    df.to_csv(f, index=False)

print(f"Данные сохранены в {csv_filename}")
df.head(10)